# Stage 1 — Raw Data Fetch (TMDB)

Rakuten TV content discovery prototype — **Stage 1 only**: automated fetch of raw metadata
from the TMDB API for the seed list in `data/content_sample.csv` (21 items, movies + TV series).

Output: `data/fetched_raw.json`, one record per seed item, ready to be consumed by Stage 2
(AI enrichment). This notebook does not do any AI enrichment — it only collects raw TMDB data.

## I. Import

### A. Libraries

In [12]:
import os
import json
import time
from pathlib import Path

import requests
import pandas as pd
from dotenv import load_dotenv

### B. Constants & config

In [13]:
load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY")
if not TMDB_API_KEY:
    raise RuntimeError(
        "TMDB_API_KEY is not set. Copy .env.example to .env and fill in your TMDB API key."
    )

TMDB_BASE_URL = "https://api.themoviedb.org/3"

YEAR_TOLERANCE = 1  # +/- years allowed between seed list year and TMDB result year

INPUT_CSV_PATH = Path("data/content_sample.csv")
OUTPUT_JSON_PATH = Path("data/fetched_raw.json")

REQUEST_TIMEOUT = 10  # seconds
MAX_RETRIES = 2

## II. Data preparation

### A. Chargement du seed list

In [14]:
EXPECTED_COLUMNS = {"content_id", "title", "year"}


def load_seed_list(csv_path: Path) -> pd.DataFrame:
    """Load the seed list CSV and validate that the expected columns are present."""
    df = pd.read_csv(csv_path)

    missing_columns = EXPECTED_COLUMNS - set(df.columns)
    if missing_columns:
        raise ValueError(f"Seed list is missing expected columns: {missing_columns}")

    df["content_id"] = df["content_id"].astype(str)
    df["year"] = df["year"].astype(int)

    return df


seed_df = load_seed_list(INPUT_CSV_PATH)
seed_df.head()

,content_id,title,year
0,1001,The Shawshank Redemption,1994
1,1002,Inception,2010
2,1003,Parasite,2019
3,1004,The Grand Budapest Hotel,2014
4,1005,Planet Earth II,2016


### B. Fonctions de recherche TMDB

In [15]:
def search_tmdb(title: str, media_type: str) -> list:
    """
    Query the TMDB search endpoint for a given title.
    media_type: "movie" or "tv".
    Returns the raw list of results (empty list on failure or no results).
    """
    if media_type not in ("movie", "tv"):
        raise ValueError(f"Unsupported media_type: {media_type}")

    url = f"{TMDB_BASE_URL}/search/{media_type}"
    params = {"api_key": TMDB_API_KEY, "query": title}

    for attempt in range(MAX_RETRIES + 1):
        try:
            response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            return response.json().get("results", [])
        except requests.exceptions.RequestException as e:
            if attempt == MAX_RETRIES:
                print(f"[search_tmdb] Failed for '{title}' ({media_type}): {e}")
                return []
            time.sleep(0.5)

    return []


def extract_year_from_result(result: dict, media_type: str):
    """Extract the release year from a TMDB search result, or None if unavailable."""
    date_field = "release_date" if media_type == "movie" else "first_air_date"
    date_value = result.get(date_field)

    if not date_value:
        return None

    try:
        return int(date_value[:4])
    except (ValueError, TypeError):
        return None

### C. Fonction de matching

In [16]:
def find_best_match(title: str, target_year: int) -> dict:
    """
    Search TMDB for both movie and tv candidates and pick the best match against target_year.
    Returns a dict with keys: media_type, tmdb_id, match_confidence, year_diff.
    """
    candidates = []

    for media_type in ("movie", "tv"):
        for result in search_tmdb(title, media_type):
            year = extract_year_from_result(result, media_type)
            if year is None:
                continue
            candidates.append({
                "media_type": media_type,
                "tmdb_id": result.get("id"),
                "year_diff": abs(year - target_year),
                "popularity": result.get("popularity", 0),
            })

    if not candidates:
        return {
            "media_type": None,
            "tmdb_id": None,
            "match_confidence": "not_found",
            "year_diff": None,
        }

    within_tolerance = [c for c in candidates if c["year_diff"] <= YEAR_TOLERANCE]

    if within_tolerance:
        best = min(within_tolerance, key=lambda c: c["year_diff"])
        confidence = "matched"
    else:
        best = max(candidates, key=lambda c: c["popularity"])
        confidence = "year_mismatch"

    return {
        "media_type": best["media_type"],
        "tmdb_id": best["tmdb_id"],
        "match_confidence": confidence,
        "year_diff": best["year_diff"],
    }

## III. Data processing

### A. Fetch des détails

In [17]:
def fetch_details(tmdb_id: int, media_type: str) -> dict:
    """
    Fetch overview, genres, runtime, original_language, cast (top 3-5) and
    director/creator for a matched TMDB item.
    """
    params = {"api_key": TMDB_API_KEY}

    details_response = requests.get(
        f"{TMDB_BASE_URL}/{media_type}/{tmdb_id}", params=params, timeout=REQUEST_TIMEOUT
    )
    details_response.raise_for_status()
    details = details_response.json()

    credits_response = requests.get(
        f"{TMDB_BASE_URL}/{media_type}/{tmdb_id}/credits", params=params, timeout=REQUEST_TIMEOUT
    )
    credits_response.raise_for_status()
    credits = credits_response.json()

    cast = [member["name"] for member in credits.get("cast", [])[:5]]

    if media_type == "movie":
        director = next(
            (member["name"] for member in credits.get("crew", []) if member.get("job") == "Director"),
            None,
        )
        runtime = details.get("runtime")
    else:
        creators = details.get("created_by", [])
        director = ", ".join(creator["name"] for creator in creators) if creators else None
        episode_run_times = details.get("episode_run_time") or [None]
        runtime = episode_run_times[0]

    return {
        "overview": details.get("overview"),
        "genres": [genre["name"] for genre in details.get("genres", [])],
        "runtime": runtime,
        "original_language": details.get("original_language"),
        "cast": cast,
        "director": director,
    }

### B. Fetch des keywords

In [18]:
def fetch_keywords(tmdb_id: int, media_type: str) -> list:
    """
    Fetch keywords for a TMDB item. Returns an empty list if unavailable,
    never raises.
    """
    url = f"{TMDB_BASE_URL}/{media_type}/{tmdb_id}/keywords"
    params = {"api_key": TMDB_API_KEY}

    try:
        response = requests.get(url, params=params, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        data = response.json()
    except requests.exceptions.RequestException:
        return []

    # movie endpoint returns {"keywords": [...]}, tv endpoint returns {"results": [...]}
    keyword_list = data.get("keywords") or data.get("results") or []
    return [kw["name"] for kw in keyword_list]

### C. Orchestration

In [19]:
OUTPUT_SCHEMA_KEYS = [
    "content_id", "title", "year", "media_type", "tmdb_id",
    "match_confidence", "year_diff", "overview", "genres", "runtime",
    "original_language", "cast", "director", "keywords", "fetch_status",
]


def _empty_detail_fields() -> dict:
    return {
        "overview": None,
        "genres": None,
        "runtime": None,
        "original_language": None,
        "cast": None,
        "director": None,
        "keywords": None,
    }


def process_title(row) -> dict:
    """
    Run the full stage-1 pipeline for a single seed list row:
    find_best_match -> fetch_details -> fetch_keywords.
    Always returns a dict with the same keys (OUTPUT_SCHEMA_KEYS), regardless
    of whether the match/fetch succeeded, so stage 2 can rely on a fixed schema.
    """
    base = {
        "content_id": row["content_id"],
        "title": row["title"],
        "year": row["year"],
    }

    try:
        match = find_best_match(row["title"], row["year"])
    except Exception as e:
        print(f"[process_title] Matching failed for '{row['title']}': {e}")
        return {
            **base,
            "media_type": None,
            "tmdb_id": None,
            "match_confidence": None,
            "year_diff": None,
            **_empty_detail_fields(),
            "fetch_status": "error",
        }

    if match["match_confidence"] == "not_found":
        return {**base, **match, **_empty_detail_fields(), "fetch_status": "not_found"}

    try:
        details = fetch_details(match["tmdb_id"], match["media_type"])
        keywords = fetch_keywords(match["tmdb_id"], match["media_type"])
        return {**base, **match, **details, "keywords": keywords, "fetch_status": "ok"}
    except Exception as e:
        print(f"[process_title] Fetch failed for '{row['title']}' (tmdb_id={match['tmdb_id']}): {e}")
        return {**base, **match, **_empty_detail_fields(), "fetch_status": "error"}

## IV. Master / Export

### A. Construction du dataset final

In [ ]:
raw_data = []

for i, row in seed_df.iterrows():
    print(f"[{i + 1}/{len(seed_df)}] Processing '{row['title']}' ({row['year']})...")
    record = process_title(row)
    raw_data.append(record)
    time.sleep(0.05)  # stay polite with TMDB's rate limit

print(f"\nDone. Processed {len(raw_data)} items.")

[1/21] Processing 'The Shawshank Redemption' (1994)...
[2/21] Processing 'Inception' (2010)...
[3/21] Processing 'Parasite' (2019)...
[4/21] Processing 'The Grand Budapest Hotel' (2014)...
[5/21] Processing 'Planet Earth II' (2016)...
[6/21] Processing 'Breaking Bad' (2008)...
[7/21] Processing 'Amélie' (2001)...
[8/21] Processing 'The Dark Knight' (2008)...
[9/21] Processing 'Moonlight' (2016)...


### B. Validation rapide

In [ ]:
total = len(raw_data)
matched = sum(1 for r in raw_data if r["fetch_status"] == "ok" and r["match_confidence"] == "matched")
year_mismatch = sum(1 for r in raw_data if r["match_confidence"] == "year_mismatch")
not_found = sum(1 for r in raw_data if r["fetch_status"] == "not_found")
errors = sum(1 for r in raw_data if r["fetch_status"] == "error")

print(f"Total items:      {total}")
print(f"Matched:          {matched}")
print(f"Year mismatch:    {year_mismatch}")
print(f"Not found:        {not_found}")
print(f"Errors:           {errors}")

if not_found or errors:
    print("\nItems needing manual review:")
    for r in raw_data:
        if r["fetch_status"] in ("not_found", "error"):
            print(f"  - {r['title']} ({r['year']}): fetch_status={r['fetch_status']}")

Total items:      21
Matched:          21
Year mismatch:    0
Not found:        0
Errors:           0


### C. Export

In [ ]:
OUTPUT_JSON_PATH.parent.mkdir(parents=True, exist_ok=True)

with open(OUTPUT_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(raw_data, f, indent=2, ensure_ascii=False)

print(f"Saved {len(raw_data)} items to {OUTPUT_JSON_PATH}")

Saved 21 items to data/fetched_raw.json
